# Modül Kodları (Notebook içi)

Bu notebook, `src/` altındaki modüllerin aynısını hücreler halinde içerir. Diğer notebooklarda bu hücreleri çalıştırarak fonksiyonları kullanabilirsiniz.

In [1]:
# data.py
from __future__ import annotations
from typing import Tuple
import numpy as np


def make_synthetic_linear(
    n: int,
    d: int,
    p: int,
    noise_std: float = 0.5,
    seed: int | None = None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    x1 = rng.normal(size=(n, 1))
    features = [x1]
    if d >= 2:
        x2 = x1 + (10.0 ** (-p)) * rng.normal(size=(n, 1))
        features.append(x2)
    if d > 2:
        rest = rng.normal(size=(n, d - 2))
        features.append(rest)
    X = np.hstack(features)
    theta_true = rng.normal(size=(d,))
    noise = noise_std * rng.normal(size=n)
    y = X @ theta_true + noise
    return X, y, theta_true


def train_val_split(
    X: np.ndarray,
    y: np.ndarray,
    val_ratio: float = 0.2,
    seed: int | None = None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    n = X.shape[0]
    idx = rng.permutation(n)
    split = int(n * (1 - val_ratio))
    train_idx, val_idx = idx[:split], idx[split:]
    return X[train_idx], X[val_idx], y[train_idx], y[val_idx]


def standardize(
    X_train: np.ndarray, X_val: np.ndarray
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0) + 1e-12
    X_train_std = (X_train - mean) / std
    X_val_std = (X_val - mean) / std
    return X_train_std, X_val_std, mean, std


In [2]:
# solvers.py
from dataclasses import dataclass
from typing import Callable, Dict, List, Tuple
import numpy as np
from numpy.linalg import inv, solve
from scipy.linalg import lstsq


def normal_eq_inverse(X: np.ndarray, y: np.ndarray) -> np.ndarray:
    xtx = X.T @ X
    xty = X.T @ y
    return inv(xtx) @ xty


def normal_eq_solve(X: np.ndarray, y: np.ndarray) -> np.ndarray:
    xtx = X.T @ X
    xty = X.T @ y
    return solve(xtx, xty)


def least_squares_qr(X: np.ndarray, y: np.ndarray) -> np.ndarray:
    theta, *_ = lstsq(X, y)
    return theta


@dataclass
class GDRuntimeLog:
    losses: List[float]
    grad_norms: List[float]
    thetas: List[np.ndarray]


def gradient_descent(
    X: np.ndarray,
    y: np.ndarray,
    alpha: float = 1e-3,
    max_iter: int = 10_000,
    tol: float = 1e-6,
    log_every: int = 10,
    callback: Callable[[int, float, float], None] | None = None,
) -> Tuple[np.ndarray, Dict[str, List[float]]]:
    n, d = X.shape
    theta = np.zeros(d)
    losses: List[float] = []
    grad_norms: List[float] = []
    for k in range(1, max_iter + 1):
        residual = X @ theta - y
        loss = 0.5 / n * np.dot(residual, residual)
        grad = (X.T @ residual) / n
        grad_norm = np.linalg.norm(grad)
        theta -= alpha * grad
        if k % log_every == 0 or k == 1:
            losses.append(loss)
            grad_norms.append(grad_norm)
            if callback:
                callback(k, loss, grad_norm)
        if grad_norm < tol:
            break
    history = {"losses": losses, "grad_norms": grad_norms}
    return theta, history


In [3]:
# gradcheck.py
from typing import Callable, Iterable, Tuple
import numpy as np


def J_mse(theta: np.ndarray, X: np.ndarray, y: np.ndarray) -> float:
    n = X.shape[0]
    residual = X @ theta - y
    return 0.5 / n * np.dot(residual, residual)


def grad_mse_analytic(theta: np.ndarray, X: np.ndarray, y: np.ndarray) -> np.ndarray:
    n = X.shape[0]
    return (X.T @ (X @ theta - y)) / n


def grad_numeric_central(
    J: Callable[[np.ndarray], float],
    theta: np.ndarray,
    eps: float = 1e-4,
) -> np.ndarray:
    grad = np.zeros_like(theta)
    for i in range(theta.size):
        e = np.zeros_like(theta)
        e[i] = 1.0
        grad[i] = (J(theta + eps * e) - J(theta - eps * e)) / (2 * eps)
    return grad


def epsilon_sweep(
    theta: np.ndarray,
    X: np.ndarray,
    y: np.ndarray,
    eps_list: Iterable[float],
) -> Tuple[np.ndarray, np.ndarray]:
    analytic = grad_mse_analytic(theta, X, y)
    eps_array = np.array(list(eps_list))
    relerrs = []
    for eps in eps_array:
        numeric = grad_numeric_central(lambda t: J_mse(t, X, y), theta, eps)
        denom = np.maximum(np.abs(analytic), np.abs(numeric))
        relerr = np.linalg.norm((analytic - numeric) / np.maximum(denom, 1e-15))
        relerrs.append(relerr)
    return eps_array, np.array(relerrs)


In [4]:
# metrics.py
import numpy as np
from numpy.linalg import cond


def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def cond_xtx(X: np.ndarray) -> float:
    return float(cond(X.T @ X))


In [6]:
# utils.py
import contextlib
import time
import numpy as np
from typing import Iterator


def set_seed(seed: int | None = None) -> np.random.Generator:
    return np.random.default_rng(seed)


@contextlib.contextmanager
def elapsed_timer() -> Iterator[callable]:
    start = time.perf_counter()
    def _elapsed():
        return time.perf_counter() - start
    yield _elapsed
